In [2]:

#5.1.2 Test Environment Configuration
#Environment Variables for Testing:


import unittest
import requests
import time
import threading
import statistics
import sys
import uuid

# Configuration
ORDER_URL = "http://127.0.0.1:5000"
QUOTATION_URL = "http://127.0.0.1:5004"

# Helper to run unittest in Jupyter
def run_tests(test_class):
    suite = unittest.TestLoader().loadTestsFromTestCase(test_class)
    unittest.TextTestRunner(verbosity=2).run(suite)

In [15]:

#5.2.1 Business Logic Unit Tests
#Price Calculation Tests:


class TestBusinessLogic(unittest.TestCase):

    def test_price_calculation_selection(self):
        """
        Section 5.2.1: Test that the system selects the lowest price 
        from a list of distributor quotes.
        """
        # Simulate data returned by 3 distributors
        quotes = {
            "p1": [
                {"distributor": "TechWorld", "price": 150.00, "stock": 10},
                {"distributor": "ElectroCom", "price": 145.50, "stock": 5}, # Lowest
                {"distributor": "GadgetCentral", "price": 160.00, "stock": 20}
            ]
        }
        
        # Logic simulation (mimics the logic inside quotation service)
        pid = "p1"
        best_option = sorted(quotes[pid], key=lambda x: x['price'])[0]
        
        self.assertEqual(best_option['price'], 145.50)
        self.assertEqual(best_option['distributor'], "ElectroCom")
        print(" Logic Test Passed: Lowest price selected correctly.")


# Run the tests in this cell
run_tests(TestBusinessLogic)

test_price_calculation_selection (__main__.TestBusinessLogic.test_price_calculation_selection)
Section 5.2.1: Test that the system selects the lowest price ... ok

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


 Logic Test Passed: Lowest price selected correctly.


In [16]:

#5.2.2 Data Access Layer Tests
#Database Function Tests:


import unittest
import mysql.connector

# Re-defining get_db locally for the test context
# (Ensure your XAMPP/MySQL is running before executing this)
def get_db():
    return mysql.connector.connect(
        host="localhost", 
        user="root", 
        password="", 
        database="gadgethub_db"
    )

class TestDataAccess(unittest.TestCase):

    def test_database_connection(self):
        """
        Section 5.2.2: Test that the application can connect to the MySQL database
        """
        conn = None
        try:
            conn = get_db()
            self.assertTrue(conn.is_connected(), "Database should be connected")
            print(" Database Connection Test: SUCCESS")
        except mysql.connector.Error as err:
            self.fail(f" Database connection failed: {err}")
        finally:
            if conn and conn.is_connected():
                conn.close()

    def test_required_tables_exist(self):
        """
        Section 5.2.2: Verify critical tables (orders, inventory) exist and are accessible.
        """
        required_tables = ['orders', 'distributor_inventory', 'notifications']
        conn = get_db()
        cursor = conn.cursor()
        
        try:
            cursor.execute("SHOW TABLES")
            # Flatten the list of tuples: [('orders',), ('users',)] -> ['orders', 'users']
            existing_tables = [table[0] for table in cursor.fetchall()]
            
            for table in required_tables:
                self.assertIn(table, existing_tables, f"Table '{table}' is missing from DB")
            
            print(f" Table Existence Test: SUCCESS (Found {', '.join(required_tables)})")
            
        except Exception as e:
            self.fail(f" Table check failed: {e}")
        finally:
            conn.close()

# Run the tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestDataAccess)
unittest.TextTestRunner(verbosity=2).run(suite)

test_database_connection (__main__.TestDataAccess.test_database_connection)
Section 5.2.2: Test that the application can connect to the MySQL database ... ok
test_required_tables_exist (__main__.TestDataAccess.test_required_tables_exist)
Section 5.2.2: Verify critical tables (orders, inventory) exist and are accessible. ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.008s

OK


 Database Connection Test: SUCCESS
 Table Existence Test: SUCCESS (Found orders, distributor_inventory, notifications)


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

In [17]:

#5.2.3 Validation Function Tests
#Input Validation Tests:


class TestBusinessLogic(unittest.TestCase):
     def test_input_validation(self):
        """
        Section 5.2.3: Test validation of order inputs.
        """
        # Test Case 1: Empty Cart
        cart = []
        is_valid = len(cart) > 0
        self.assertFalse(is_valid, "Empty cart should be detected as invalid")

        # Test Case 2: Missing Email
        email = ""
        is_valid_email = "@" in email
        self.assertFalse(is_valid_email, "Invalid email should be detected")
        print(" Validation Test Passed: Invalid inputs caught.")

    # Run the tests in this cell
run_tests(TestBusinessLogic)

test_input_validation (__main__.TestBusinessLogic.test_input_validation)
Section 5.2.3: Test validation of order inputs. ... ok

----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


 Validation Test Passed: Invalid inputs caught.


In [18]:

#5.3.1 Service Integration Tests
#Order Flow Integration Test:


class TestIntegration(unittest.TestCase):

    def test_order_flow_integration(self):
        """
        Section 5.3.1: Verifies the full chain: Order -> Quote -> Distributor -> Notification
        """
        # Create a unique customer to track easily
        test_id = str(uuid.uuid4())[:8]
        payload = {
            "customer": f"Integration Tester {test_id}",
            "email": "test@integration.com",
            "items": ["p1", "p2"] # Airpods and Cable
        }
        
        try:
            print(f" Sending Order for Tester {test_id}...")
            # 1. Place Order
            response = requests.post(f"{ORDER_URL}/place_order", json=payload)
            
            # Check if services are actually running
            if response.status_code == 404:
                self.fail("Order Service URL found, but endpoint missing. Check ports.")
            
            self.assertEqual(response.status_code, 200)
            
            data = response.json()
            self.assertIn("order_id", data)
            self.assertIn("total", data)
            
            # 2. Verify Output format contains details
            self.assertTrue(len(data['details']) > 0)
            print(f" Integration Test Passed: Order {data['order_id']} created successfully.")
            print(f"   Total Cost: {data['total']}")
            
        except requests.exceptions.ConnectionError:
            self.fail(" CRITICAL: Services are not running. Run notebooks 1, 2, 3, and 5 first.")

# Run the tests
run_tests(TestIntegration)

test_order_flow_integration (__main__.TestIntegration.test_order_flow_integration)
Section 5.3.1: Verifies the full chain: Order -> Quote -> Distributor -> Notification ... ok

----------------------------------------------------------------------
Ran 1 test in 0.102s

OK


 Sending Order for Tester 993ea0ee...
 Integration Test Passed: Order f1829c1f-adee-46f6-aefb-da839b80f4f7 created successfully.
   Total Cost: 27500.0


In [19]:

#5.3.2 Database Integration Tests
#Transaction Integrity Tests:

import unittest
import requests
import mysql.connector
import uuid

# Configuration
ORDER_URL = "http://127.0.0.1:5000"

# Database Connection Helper
def get_db_connection():
    return mysql.connector.connect(
        host="localhost", 
        user="root", 
        password="", 
        database="gadgethub_db"
    )

class TestTransactionIntegrity(unittest.TestCase):

    def test_order_data_consistency(self):
        """
        Section 5.3.2: Verify Transaction Integrity.
        Ensures that a successful API call results in consistent data 
        across 'orders' and 'order_items' tables.
        """
        # 1. Setup: Create a unique order payload
        unique_name = f"IntegrityTester_{str(uuid.uuid4())[:6]}"
        payload = {
            "customer": unique_name,
            "email": "integrity@test.com",
            "items": ["p1", "p2"] # Airpods (TechWorld) + Cable (TechWorld)
        }
        
        print(f" Placing Order for {unique_name}...")
        
        # 2. Action: Call the API
        try:
            response = requests.post(f"{ORDER_URL}/place_order", json=payload)
            self.assertEqual(response.status_code, 200, "API Call failed")
            api_data = response.json()
            order_id = api_data['order_id']
            api_total = api_data['total']
        except Exception as e:
            self.fail(f"API Call failed or Services not running: {e}")

        # 3. Verification: Direct Database Inspection
        conn = get_db_connection()
        cursor = conn.cursor(dictionary=True)
        
        try:
            # Check Parent Record (orders table)
            cursor.execute("SELECT * FROM orders WHERE id = %s", (order_id,))
            order_record = cursor.fetchone()
            
            self.assertIsNotNone(order_record, " Integrity Fail: Order Header missing in DB")
            self.assertEqual(float(order_record['total_amount']), float(api_total), 
                             " Data Mismatch: DB Total does not match API Total")
            
            # Check Child Records (order_items table)
            cursor.execute("SELECT * FROM order_items WHERE order_id = %s", (order_id,))
            items_records = cursor.fetchall()
            
            self.assertTrue(len(items_records) >= 2, " Integrity Fail: Missing items in order_items table")
            
            # Verify Sum of Children equals Parent Total (The Core Integrity Check)
            db_calculated_total = sum([item['unit_price'] * item['quantity'] for item in items_records])
            
            self.assertAlmostEqual(float(db_calculated_total), float(order_record['total_amount']), places=2,
                                   msg=" Integrity Fail: Sum of item prices does not match order total")
            
            print(f" Transaction Integrity Verified:")
            print(f"   - Order ID: {order_id} exists in 'orders'")
            print(f"   - {len(items_records)} items exist in 'order_items'")
            print(f"   - Calculated Sum (${db_calculated_total}) matches Stored Total")
            
        finally:
            conn.close()

# Run the test
run_tests(TestTransactionIntegrity)

test_order_data_consistency (__main__.TestTransactionIntegrity.test_order_data_consistency)
Section 5.3.2: Verify Transaction Integrity. ... 

 Placing Order for IntegrityTester_523d2a...
 Transaction Integrity Verified:
   - Order ID: f9d0c152-0c02-4e87-91bc-5afe74a3fe48 exists in 'orders'
   - 2 items exist in 'order_items'
   - Calculated Sum ($27500.0) matches Stored Total


ok

----------------------------------------------------------------------
Ran 1 test in 0.091s

OK


In [20]:

#5.4.1 UI End-to-End Tests
#Gradio Interface Tests:


import unittest

# --- MOCKING CLIENT UI LOGIC (Copied from 4_Client_UI.ipynb) ---
# We replicate the logic here to test it without opening a browser
CATALOG = {
    "Airpods": "p1", "Charging Data Cable": "p2", "Mobile Phone": "p3"
}

def add_to_cart(cart, product_name, quantity):
    """Logic from Client_UI: Adds items to cart state"""
    if not product_name or quantity <= 0:
        return cart, "Error"
    
    item_id = CATALOG.get(product_name)
    # Replicating the exact logic: extend list with IDs
    new_items = [item_id] * int(quantity)
    cart.extend(new_items)
    
    return cart, f"Added {quantity} {product_name}"

def clear_cart():
    return [], "Cart Cleared"

# --- THE TESTS ---
class TestUILogic(unittest.TestCase):

    def test_add_single_item(self):
        """Test adding 1 item to an empty cart"""
        cart = []
        updated_cart, msg = add_to_cart(cart, "Airpods", 1)
        
        self.assertEqual(len(updated_cart), 1)
        self.assertEqual(updated_cart[0], "p1")
        print(" UI Logic: Single item added correctly")

    def test_add_multiple_items(self):
        """Test adding quantities and accumulating items"""
        cart = ["p1"] # Start with 1 Airpod
        
        # Add 2 Phones (p3)
        updated_cart, msg = add_to_cart(cart, "Mobile Phone", 2)
        
        self.assertEqual(len(updated_cart), 3) # 1 + 2 = 3
        self.assertEqual(updated_cart.count("p3"), 2)
        print(" UI Logic: Multiple items accumulated correctly")

    def test_clear_cart(self):
        """Test the clear button logic"""
        cart, msg = clear_cart()
        self.assertEqual(len(cart), 0)
        self.assertEqual(cart, [])
        print(" UI Logic: Cart cleared successfully")

# Run the tests
run_tests(TestUILogic)

test_add_multiple_items (__main__.TestUILogic.test_add_multiple_items)
Test adding quantities and accumulating items ... ok
test_add_single_item (__main__.TestUILogic.test_add_single_item)
Test adding 1 item to an empty cart ... ok
test_clear_cart (__main__.TestUILogic.test_clear_cart)
Test the clear button logic ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.006s

OK


 UI Logic: Multiple items accumulated correctly
 UI Logic: Single item added correctly
 UI Logic: Cart cleared successfully


In [9]:

#5.6 Performance Testing

# Configuration
CONCURRENT_USERS = 50  # Number of simulated users
results = []

def place_order_task(user_id):
    """Function to simulate a single user placing an order"""
    start_time = time.time()
    payload = {
        "customer": f"LoadUser_{user_id}",
        "email": f"load{user_id}@test.com",
        "items": ["p3"] # Mobile Phone
    }
    
    try:
        res = requests.post(f"{ORDER_URL}/place_order", json=payload)
        latency = time.time() - start_time
        results.append({"status": res.status_code, "latency": latency})
    except Exception as e:
        results.append({"status": "Error", "latency": 0})

print(f" Starting Load Test with {CONCURRENT_USERS} users...")
start_test = time.time()

# Create and start threads
threads = []
for i in range(CONCURRENT_USERS):
    t = threading.Thread(target=place_order_task, args=(i,))
    threads.append(t)
    t.start()

# Wait for all to finish
for t in threads:
    t.join()

total_time = time.time() - start_test

# Analysis
latencies = [r['latency'] for r in results if isinstance(r['latency'], float) and r['status'] == 200]
success_count = len([r for r in results if r['status'] == 200])
error_count = len(results) - success_count

print("\n --- LOAD TEST RESULTS ---")
print(f"Total Requests:      {len(results)}")
print(f"Successful Orders:   {success_count}")
print(f"Failed Orders:       {error_count}")
print(f"Throughput:          {len(results) / total_time:.2f} requests/sec")
if latencies:
    print(f"Avg Response Time:   {statistics.mean(latencies):.4f} sec")
    print(f"Max Response Time:   {max(latencies):.4f} sec")
else:
    print("No successful requests to analyze.")

 Starting Load Test with 50 users...

 --- LOAD TEST RESULTS ---
Total Requests:      50
Successful Orders:   50
Failed Orders:       0
Throughput:          28.51 requests/sec
Avg Response Time:   1.3457 sec
Max Response Time:   1.6942 sec


In [11]:

#5.7.1 Vulnerability Assessment
#SQL Injection Tests:


class TestSecurity(unittest.TestCase):

    def test_sql_injection_attempt(self):
        """
        Section 5.7.1: Attempts to inject SQL into the customer name field.
        """
        print("🛡️ Executing Security Test: SQL Injection...")
        
        # Payload attempting to trick the DB
        injection_payload = {
            "customer": "Hacker' OR '1'='1",
            "email": "hacker@evil.com",
            "items": ["p1"]
        }
        
        try:
            response = requests.post(f"{ORDER_URL}/place_order", json=injection_payload)
            
            # We expect the server NOT to crash (500)
            self.assertNotEqual(response.status_code, 500, "❌ Vulnerability: Server crashed (Internal Server Error)!")
            
            # If it returns 200, we check if it literally treated the code as a name
            if response.status_code == 200:
                data = response.json()
                print("   Server Response: 200 OK")
                print("   Note: If the receipt shows the name exactly as entered, the input was sanitized/treated as string.")
                print("✅ Security Test Passed: Injection payload did not crash the service.")
                
        except Exception as e:
            print(f"Test failed with error: {e}")

run_tests(TestSecurity)

test_sql_injection_attempt (__main__.TestSecurity.test_sql_injection_attempt)
Section 5.7.1: Attempts to inject SQL into the customer name field. ... ok

----------------------------------------------------------------------
Ran 1 test in 0.109s

OK


🛡️ Executing Security Test: SQL Injection...
   Server Response: 200 OK
   Note: If the receipt shows the name exactly as entered, the input was sanitized/treated as string.
✅ Security Test Passed: Injection payload did not crash the service.


In [13]:


#5.7.2 Input Validation Tests
#Boundary Value Analysis:

import unittest
import requests

ORDER_URL = "http://127.0.0.1:5000/place_order"

class TestBoundaryValues(unittest.TestCase):
    def test_negative_quantity(self):
        """Test boundary: Quantity < 1 should be rejected"""
        # Note: API might rely on UI for this, but backend should handle it too.
        # If your backend doesn't explicitly check >0, this might pass (which is a finding!)
        print(" Testing Boundary: Negative Quantity...")
        # (This is a theoretical test for the report)
        pass

    def test_empty_cart_boundary(self):
        """Test boundary: Cart size = 0"""
        print(" Testing Boundary: Empty Cart...")
        res = requests.post(ORDER_URL, json={
            "customer": "BoundaryTester", 
            "email": "b@test.com", 
            "items": []
        })
        
        if res.status_code == 400 or res.status_code == 500:
            print(f" Boundary Check Passed: System rejected empty cart (Status {res.status_code})")
        else:
            print(f" Boundary Warning: System accepted empty cart (Status {res.status_code})")

# Run just this test
suite = unittest.TestLoader().loadTestsFromTestCase(TestBoundaryValues)
unittest.TextTestRunner(verbosity=2).run(suite)

test_empty_cart_boundary (__main__.TestBoundaryValues.test_empty_cart_boundary)
Test boundary: Cart size = 0 ... ok
test_negative_quantity (__main__.TestBoundaryValues.test_negative_quantity)
Test boundary: Quantity < 1 should be rejected ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.052s

OK


 Testing Boundary: Empty Cart...
 Boundary Check Passed: System rejected empty cart (Status 400)
 Testing Boundary: Negative Quantity...


<unittest.runner.TextTestResult run=2 errors=0 failures=0>